# Optuna Sweep on Full Data (Kaggle)

**Phase 12** — runs the Optuna hyperparameter sweep on the full training set, then retrains with the best params and writes `submission_lgbm_v2.csv`.

## How to run on Kaggle

1. Create a new notebook on Kaggle.
2. Add the competition dataset (`playground-series-s6e9`).
3. Upload `src/` as a Kaggle Dataset (e.g., slug `ev-purchase-src`).
4. Copy-paste the cells below into the notebook.
5. Run all cells.

## Expected runtime

- 30 trials × 5-fold CV on full data: **15-25 min on Kaggle CPU**.
- Each trial uses up to 1500 boost rounds with early stopping (100 rounds patience).

## Output

- `/kaggle/working/submission_lgbm_v2.csv` — Kaggle submission (286,571 rows).
- `/kaggle/working/best_params_lgbm_v2.json` — best hyperparameters.
- `/kaggle/working/run_metrics_v2.json` — full run metrics (CV AUC, fold AUCs, runtime).
- `/kaggle/working/mlruns/` — MLflow tracking directory (zipped at the end as a notebook output).

## Upload the submission

After the notebook finishes, go to **Output** → `submission_lgbm_v2.csv` → **Submit to Competition**.

In [ ]:
import os
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
# Cell 1: Install Optuna and confirm src/ is importable
!pip install --quiet optuna

import os
import sys
from pathlib import Path

# Add the `src/` directory to sys.path. Adjust KAGGLE_SRC to match
# your uploaded dataset slug.
KAGGLE_SRC = "/kaggle/input/ev-purchase-src/src"
if os.path.isdir(KAGGLE_SRC):
    sys.path.insert(0, os.path.dirname(KAGGLE_SRC))
    print(f"Added {KAGGLE_SRC} to sys.path")
else:
    sys.path.insert(0, "/kaggle/working")
    print(f"Using {sys.path[0]}")

import optuna
print(f"optuna {optuna.__version__}")
from src.config import (
    CATEGORICAL_COLS, FEATURE_COLS, N_FOLDS, RANDOM_SEED,
    SAMPLE_SUBMISSION_CSV, TARGET_COL, TEST_CSV, TRAIN_CSV,
)
from src.cv import make_folds
from src.data import load_data
from src.features import build_features
from src.predict import make_submission
from src.train_lgbm import train_lgbm
from src.tune_lgbm import run_optuna_sweep
from src import tracking

# On Kaggle, the competition data lives at /kaggle/input/playground-series-s6e9/.
KAGGLE_DATA = Path("/kaggle/input/playground-series-s6e9")
TRAIN_CSV = KAGGLE_DATA / "train.csv"
TEST_CSV = KAGGLE_DATA / "test.csv"
SAMPLE_SUBMISSION_CSV = KAGGLE_DATA / "sample_submission.csv"
print(f"Train: {TRAIN_CSV}  size: {TRAIN_CSV.stat().st_size / 1e6:.1f} MB")
print(f"Test:  {TEST_CSV}  size: {TEST_CSV.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Cell 2: Set up MLflow on a writable directory
import shutil
from pathlib import Path

import mlflow

MLRUNS_DIR = Path("/kaggle/working/mlruns")
if MLRUNS_DIR.exists():
    shutil.rmtree(MLRUNS_DIR)
MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
tracking.set_tracking_uri(MLRUNS_DIR)
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

In [ ]:
# Cell 3: Load + features + folds
print("Loading data...")
train_raw, test_raw = load_data(TRAIN_CSV, TEST_CSV)
print(f"  train: {train_raw.shape}")
print(f"  test:  {test_raw.shape}")

print("Building features...")
train_feat = build_features(train_raw)
test_feat = build_features(test_raw)
print(f"  features: {len(FEATURE_COLS)} columns")

print("Building CV folds...")
folds = make_folds(
    train_feat[TARGET_COL].to_numpy(), n_splits=N_FOLDS, seed=RANDOM_SEED
)
print(f"  fold sizes: {__import__('numpy').bincount(folds).tolist()}")

In [ ]:
# Cell 4: Run the Optuna sweep (30 trials, 5-fold CV on full data)
import time
t0 = time.time()

N_TRIALS = 30
NUM_BOOST_ROUND = 1500
EARLY_STOPPING = 100

best_params, best_score, study = run_optuna_sweep(
    train=train_feat,
    test=test_feat,
    folds=folds,
    feature_cols=FEATURE_COLS,
    target_col=TARGET_COL,
    categorical_cols=CATEGORICAL_COLS,
    n_trials=N_TRIALS,
    num_boost_round=NUM_BOOST_ROUND,
    early_stopping_rounds=EARLY_STOPPING,
    tracking_enabled=True,
    sweep_run_name="kaggle_optuna_v2",
    seed=RANDOM_SEED,
)

print("=" * 60)
print(f"Sweep done in {time.time() - t0:.1f}s")
print(f"Best CV AUC: {best_score:.5f}")
print(f"Best params:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print("=" * 60)

In [ ]:
# Cell 5: Retrain with the best params (full data, 5-fold CV, write submission)
from sklearn.metrics import roc_auc_score

print("Retraining with best params...")
oof, test_pred, metrics = train_lgbm(
    train=train_feat,
    test=test_feat,
    folds=folds,
    feature_cols=FEATURE_COLS,
    target_col=TARGET_COL,
    params=best_params,
    num_boost_round=NUM_BOOST_ROUND,
    early_stopping_rounds=EARLY_STOPPING,
    categorical_cols=CATEGORICAL_COLS,
    tracking_enabled=True,
    run_name="kaggle_optuna_v2_retrain",
)
pooled = roc_auc_score(train_feat[TARGET_COL].to_numpy(), oof)
print(f"Final mean CV AUC: {metrics['cv_auc_mean']:.5f}  (std {metrics['cv_auc_std']:.5f})")
print(f"Final pooled OOF AUC: {pooled:.5f}")
print(f"Per-fold AUC: {[round(a, 5) for a in metrics['fold_aucs']]}")

In [ ]:
# Cell 6: Write submission
submission_path = Path("/kaggle/working/submission_lgbm_v2.csv")
make_submission(
    test_ids=test_feat["id"],
    test_pred=test_pred,
    template_path=SAMPLE_SUBMISSION_CSV,
    out_path=submission_path,
)
sub = __import__("pandas").read_csv(submission_path)
print(f"Wrote: {submission_path}  shape: {sub.shape}")
print(f"  Will_Buy_EV range: [{sub['Will_Buy_EV'].min():.4f}, {sub['Will_Buy_EV'].max():.4f}]")
print(f"  Will_Buy_EV mean:  {sub['Will_Buy_EV'].mean():.4f}")

In [ ]:
# Cell 7: Persist best params and metrics as JSON
import json
from pathlib import Path

working = Path("/kaggle/working")
(working / "best_params_lgbm_v2.json").write_text(json.dumps(best_params, indent=2))
(working / "run_metrics_v2.json").write_text(json.dumps({
    "best_params": best_params,
    "best_score_study": best_score,
    "final_cv_auc_mean": metrics["cv_auc_mean"],
    "final_cv_auc_std": metrics["cv_auc_std"],
    "final_pooled_oof_auc": pooled,
    "fold_aucs": metrics["fold_aucs"],
    "n_trials": N_TRIALS,
}, indent=2))
print("best_params_lgbm_v2.json and run_metrics_v2.json written")
print(f"\nAll outputs in /kaggle/working/:")
for p in sorted(working.iterdir()):
    if p.name.startswith("."):
        continue
    size = p.stat().st_size if p.is_file() else 0
    print(f"  {p.name}  ({size / 1e6:.1f} MB)")

In [ ]:
# Cell 8: Zip the MLflow runs so they appear as a notebook output
import shutil
from pathlib import Path

working = Path("/kaggle/working")
mlruns_zip = working / "mlruns.zip"
if mlruns_zip.exists():
    mlruns_zip.unlink()
shutil.make_archive(
    base_name=str(working / "mlruns"),
    format="zip",
    root_dir=str(working),
    base_dir="mlruns",
)
print(f"MLflow zipped: {mlruns_zip}")
print(f"Size: {mlruns_zip.stat().st_size / 1024:.1f} KB")

## Next steps

1. **Submit** `/kaggle/working/submission_lgbm_v2.csv` to the competition.
2. **Download** `/kaggle/working/best_params_lgbm_v2.json` and `/kaggle/working/run_metrics_v2.json` and paste the values into the `docs/experiment_log.md` Experiment 5 entry.
3. **Inspect** the MLflow runs in `/kaggle/working/mlruns/` (download the .zip, extract, run `mlflow ui` locally).
4. **If the gap to the top score is still > 0.003**, consider Phase 13 (XGBoost + CatBoost diversity) followed by Phase 14 (rank-averaged ensemble).